# DL LAB 6 – Backpropagation for Handwritten Digit Classification (MNIST)
### Name: Naman Bansal | Roll: 102303496

This notebook implements a **neural network from scratch** using the backpropagation algorithm
to classify handwritten digits from the **MNIST dataset** (0–9).

**Architecture:** Input (784) → Hidden (128, ReLU) → Hidden (64, ReLU) → Output (10, Softmax)

**Key components implemented manually:**
- Forward propagation with ReLU and Softmax
- Cross-entropy loss
- Backward propagation (gradient computation via chain rule)
- Mini-batch Stochastic Gradient Descent (SGD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

np.random.seed(42)
print("Libraries imported.")

## 1. Load and Preprocess MNIST Dataset

In [ ]:
# Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X, y = mnist.data, mnist.target.astype(int)

# Normalize pixel values to [0, 1]
X = X / 255.0

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# One-hot encode labels
def one_hot(y, num_classes=10):
    oh = np.zeros((y.shape[0], num_classes))
    oh[np.arange(y.shape[0]), y] = 1
    return oh

y_train_oh = one_hot(y_train)
y_test_oh  = one_hot(y_test)

print(f"Training set  : X={X_train.shape}, y={y_train_oh.shape}")
print(f"Test set      : X={X_test.shape},  y={y_test_oh.shape}")

## 2. Visualize Sample Digits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {y_train[i]}", fontsize=12)
    ax.axis("off")
plt.suptitle("Sample MNIST Digits", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Activation Functions & Derivatives

In [ ]:
def relu(z):
    """ReLU activation"""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivative of ReLU"""
    return (z > 0).astype(float)

def softmax(z):
    """Softmax activation (numerically stable)"""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

print("Activation functions defined.")

## 4. Parameter Initialization
Using **He initialization** for ReLU layers to prevent vanishing/exploding gradients.

In [ ]:
def initialize_parameters(layer_dims):
    """Initialize weights using He initialization and biases to zero."""
    parameters = {}
    for l in range(1, len(layer_dims)):
        parameters[f"W{l}"] = np.random.randn(
            layer_dims[l-1], layer_dims[l]
        ) * np.sqrt(2.0 / layer_dims[l-1])
        parameters[f"b{l}"] = np.zeros((1, layer_dims[l]))
    return parameters

# Network architecture: 784 -> 128 -> 64 -> 10
layer_dims = [784, 128, 64, 10]
params = initialize_parameters(layer_dims)

print("Network architecture:", " -> ".join(str(d) for d in layer_dims))
total_params = sum(
    params[f"W{l}"].size + params[f"b{l}"].size
    for l in range(1, len(layer_dims))
)
print(f"Total trainable parameters: {total_params:,}")

## 5. Forward Propagation
Compute Z = X @ W + b, then apply activation.
Hidden layers use ReLU; output layer uses Softmax for multi-class probabilities.

In [ ]:
def forward_propagation(X, parameters, num_layers):
    """Forward pass through the network.
    Returns: output probabilities and cache of intermediate values."""
    cache = {"A0": X}
    A = X

    # Hidden layers (ReLU)
    for l in range(1, num_layers):
        Z = A @ parameters[f"W{l}"] + parameters[f"b{l}"]
        A = relu(Z)
        cache[f"Z{l}"] = Z
        cache[f"A{l}"] = A

    # Output layer (Softmax)
    Z_out = A @ parameters[f"W{num_layers}"] + parameters[f"b{num_layers}"]
    A_out = softmax(Z_out)
    cache[f"Z{num_layers}"] = Z_out
    cache[f"A{num_layers}"] = A_out

    return A_out, cache

print("Forward propagation defined.")

## 6. Cross-Entropy Loss
For multi-class classification: $L = -\frac{1}{m} \sum_{i} \sum_{k} y_{ik} \log(\hat{y}_{ik})$

In [ ]:
def compute_loss(y_true, y_pred):
    """Cross-entropy loss."""
    m = y_true.shape[0]
    # Clip to avoid log(0)
    y_pred_clipped = np.clip(y_pred, 1e-12, 1 - 1e-12)
    loss = -np.sum(y_true * np.log(y_pred_clipped)) / m
    return loss

print("Loss function defined.")

## 7. Backward Propagation (Chain Rule)
Compute gradients of the loss with respect to each weight and bias.

For **softmax + cross-entropy**, the gradient simplifies to: $dZ_{out} = \hat{y} - y$

For **hidden layers with ReLU**: $dZ_l = dA_l \odot \text{ReLU}'(Z_l)$

In [ ]:
def backward_propagation(y_true, y_pred, cache, parameters, num_layers):
    """Backward pass: compute gradients using chain rule."""
    m = y_true.shape[0]
    gradients = {}

    # Output layer gradient (softmax + cross-entropy shortcut)
    dZ = (y_pred - y_true) / m

    for l in range(num_layers, 0, -1):
        A_prev = cache[f"A{l-1}"]

        gradients[f"dW{l}"] = A_prev.T @ dZ
        gradients[f"db{l}"] = np.sum(dZ, axis=0, keepdims=True)

        if l > 1:
            dA = dZ @ parameters[f"W{l}"].T
            dZ = dA * relu_derivative(cache[f"Z{l-1}"])

    return gradients

print("Backward propagation defined.")

## 8. Parameter Update (SGD)
$\theta = \theta - \eta \cdot \nabla_{\theta} L$

In [ ]:
def update_parameters(parameters, gradients, learning_rate, num_layers):
    """Update weights and biases using gradient descent."""
    for l in range(1, num_layers + 1):
        parameters[f"W{l}"] -= learning_rate * gradients[f"dW{l}"]
        parameters[f"b{l}"] -= learning_rate * gradients[f"db{l}"]
    return parameters

print("Parameter update function defined.")

In [ ]:
def compute_accuracy(y_true_labels, y_pred_probs):
    """Compute classification accuracy."""
    predictions = np.argmax(y_pred_probs, axis=1)
    return np.mean(predictions == y_true_labels) * 100

print("Accuracy function defined.")

## 9. Training Loop (Mini-Batch SGD)
Train the network using mini-batch gradient descent.
Each epoch: shuffle data → split into mini-batches → forward → loss → backward → update.

In [ ]:
# Hyperparameters
learning_rate = 0.1
epochs = 20
batch_size = 128
num_layers = len(layer_dims) - 1   # 3 layers of weights

# Re-initialize parameters
params = initialize_parameters(layer_dims)

# Storage for plotting
train_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(epochs):
    # Shuffle training data
    perm = np.random.permutation(X_train.shape[0])
    X_shuffled = X_train[perm]
    y_shuffled = y_train_oh[perm]
    y_labels_shuffled = y_train[perm]

    epoch_loss = 0
    num_batches = 0

    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]

        # Forward propagation
        y_pred, cache = forward_propagation(X_batch, params, num_layers)

        # Compute loss
        loss = compute_loss(y_batch, y_pred)
        epoch_loss += loss
        num_batches += 1

        # Backward propagation
        grads = backward_propagation(y_batch, y_pred, cache, params, num_layers)

        # Update parameters
        params = update_parameters(params, grads, learning_rate, num_layers)

    # Epoch metrics
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)

    # Training accuracy
    y_train_pred, _ = forward_propagation(X_train, params, num_layers)
    train_acc = compute_accuracy(y_train, y_train_pred)
    train_accuracies.append(train_acc)

    # Test accuracy
    y_test_pred, _ = forward_propagation(X_test, params, num_layers)
    test_acc = compute_accuracy(y_test, y_test_pred)
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.2f}%")

## 10. Training Results – Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, epochs+1), train_losses, "b-o", linewidth=2, markersize=5)
ax1.set_title("Training Loss (Cross-Entropy)", fontsize=14, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(range(1, epochs+1), train_accuracies, "b-o", label="Train", linewidth=2, markersize=5)
ax2.plot(range(1, epochs+1), test_accuracies, "r-s", label="Test", linewidth=2, markersize=5)
ax2.set_title("Classification Accuracy", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Confusion Matrix & Per-Class Accuracy

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_test_pred_final, _ = forward_propagation(X_test, params, num_layers)
y_pred_labels = np.argmax(y_test_pred_final, axis=1)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_labels)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Confusion Matrix", fontsize=16, fontweight="bold")
ax.set_xlabel("Predicted", fontsize=13)
ax.set_ylabel("True", fontsize=13)
ax.set_xticks(range(10))
ax.set_yticks(range(10))

for i in range(10):
    for j in range(10):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color=color, fontsize=10)

plt.colorbar(im)
plt.tight_layout()
plt.show()

# Classification report
print(classification_report(y_test, y_pred_labels, digits=3))

## 12. Visualize Predictions on Test Samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
indices = np.random.choice(X_test.shape[0], 10, replace=False)

for idx, ax in zip(indices, axes.flat):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    pred = y_pred_labels[idx]
    true = y_test[idx]
    color = "green" if pred == true else "red"
    ax.set_title(f"Pred: {pred} | True: {true}",
                 fontsize=11, color=color, fontweight="bold")
    ax.axis("off")

plt.suptitle("Predictions on Test Samples\n(Green = Correct, Red = Wrong)",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 13. Misclassified Examples
Examining what the model gets wrong.

In [ ]:
misclassified = np.where(y_pred_labels != y_test)[0]
print(f"Total misclassified: {len(misclassified)} out of {len(y_test)} "
      f"({len(misclassified)/len(y_test)*100:.2f}%)")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
mis_indices = misclassified[:10]

for idx, ax in zip(mis_indices, axes.flat):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"Pred: {y_pred_labels[idx]} | True: {y_test[idx]}",
                 fontsize=11, color="red", fontweight="bold")
    ax.axis("off")

plt.suptitle("Misclassified Examples", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()